# Chapter 5. Introducing the Radio Map

The code from Chapter 5 of *Radio Propagation Ray Tracing with Sionna RT*.
The cells follow the order of the text, and the contents of each code cell are
the same as printed there.

Only the first Setup cell is not printed in the book. It performs the imports
and the scene loading that the printed code assumes. Once the setup cell has
run, the rest can be run from the top in order.

Fragments the text shows for explanation, such as a path to a scene of your own
or an object name that scene does not contain, will not run as they stand. A
note appears before each such cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

## 2. The Urban Scene Used in This Chapter

In [ ]:
from sionna.rt import load_scene, scene as scene_assets

scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9

## 3. Seeing Received Power Over an Area

In [ ]:
from sionna.rt import PlanarArray, Transmitter, RadioMapSolver

scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = RadioMapSolver()
rm = solver(scene, max_depth=3,
            cell_size=[3.0, 3.0],
            samples_per_tx=10**6)
print('Radio map shape:', rm.path_gain.shape)

## 7. A Radio Map for One Transmitter

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rm = solver(scene, max_depth=3, cell_size=[3.0, 3.0],
            samples_per_tx=10**6)
rss_dbm = 10*np.log10(rm.rss.numpy()) + 30
plt.imshow(rss_dbm[0], origin='lower', cmap='jet',
           vmin=-75, vmax=-25)
plt.colorbar(label='RSS [dBm]')

## 8. Radio Maps for Multiple Transmitters

In [ ]:
scene.remove('tx0')  # remove the transmitter from the single-Tx setup

tx_positions = [[-120.0, 60.0, 35.0],
                [  90.0, 60.0, 35.0],
                [ -15.0,-125.0, 35.0]]
for i, pos in enumerate(tx_positions):
    scene.add(Transmitter(f'tx{i}', position=pos))

rm = solver(scene, max_depth=3, cell_size=[3.0, 3.0],
            samples_per_tx=10**6)
print('Number of Tx in rm:', rm.path_gain.shape[0])

## 9. Base Station Association

In [ ]:
import numpy as np
best_tx = np.argmax(rm.rss.numpy(), axis=0)  # strongest Tx index per cell

## 10. Building a SINR Map

In [ ]:
rss = rm.rss.numpy()                      # (num_tx, H, W)
signal = rss.max(axis=0)
interf  = rss.sum(axis=0) - signal
N0 = 1e-13                                # e.g. -100 dBm
sinr_db = 10*np.log10(signal/(interf + N0))

## 11. Evaluating the Coverage Ratio

In [ ]:
threshold = 0.0  # count SINR >= 0 dB as covered
coverage = (sinr_db >= threshold).mean()
print(f'Coverage at {threshold:.0f} dB: {coverage*100:.1f} %')

## 12. Reading the Result as a CDF

In [ ]:
signal_dbm = 10*np.log10(np.maximum(signal, 1e-30)) + 30
valid = np.isfinite(sinr_db) & (signal_dbm >= -110.0)
sorted_sinr = np.sort(sinr_db[valid].flatten())
cdf = np.arange(1, len(sorted_sinr) + 1)/len(sorted_sinr)
plt.plot(sorted_sinr, cdf)

## 13. Sampling User Positions from a Radio Map

In [ ]:
path_gain = rm.path_gain.numpy()[0].flatten()
valid_idx = np.flatnonzero(np.isfinite(path_gain) & (path_gain > 0))
idx = np.random.choice(valid_idx, size=50, replace=True)

centers = rm.cell_centers.numpy().reshape(-1, 3)
positions = centers[idx]  # real-space coordinates of the chosen cells [m]